<a href="https://colab.research.google.com/github/lewliu2003-create/flight_pred_ml/blob/main/regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score,
                             mean_absolute_error, mean_squared_error, r2_score)



In [2]:
drive.mount('/content/drive')

csv_path = '/content/drive/MyDrive/flights_2026_stacked.csv'


try:
    df = pd.read_csv(csv_path)
    # display(df.head())
    display(df.columns)
except FileNotFoundError:
    print(f"Could not find the file.")

Mounted at /content/drive


Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE',
       'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'OP_CARRIER', 'TAIL_NUM',
       'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN_AIRPORT_SEQ_ID',
       'ORIGIN_CITY_MARKET_ID', 'ORIGIN', 'ORIGIN_CITY_NAME',
       'ORIGIN_STATE_ABR', 'DEST_AIRPORT_ID', 'DEST_AIRPORT_SEQ_ID',
       'DEST_CITY_MARKET_ID', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR',
       'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15',
       'DEP_DELAY_GROUP', 'DEP_TIME_BLK', 'TAXI_OUT', 'WHEELS_OFF',
       'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY',
       'ARR_DELAY_NEW', 'ARR_DEL15', 'ARR_DELAY_GROUP', 'ARR_TIME_BLK',
       'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME',
       'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'FLIGHTS', 'DISTANCE',
       'DISTANCE_GROUP', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
       'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY'],
      dtype='obj

In [3]:
display(df.head())

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,...,ACTUAL_ELAPSED_TIME,AIR_TIME,FLIGHTS,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,2026,1,1,1,4,1/1/2026 12:00:00 AM,AA,19805,AA,N101NN,...,305.0,286.0,1.0,2611.0,11,0.0,0.0,0.0,0.0,59.0
1,2026,1,1,1,4,1/1/2026 12:00:00 AM,AA,19805,AA,N101NN,...,397.0,347.0,1.0,2475.0,10,0.0,71.0,15.0,0.0,0.0
2,2026,1,1,1,4,1/1/2026 12:00:00 AM,AA,19805,AA,N103NN,...,320.0,284.0,1.0,2611.0,11,11.0,0.0,0.0,0.0,8.0
3,2026,1,1,1,4,1/1/2026 12:00:00 AM,AA,19805,AA,N103NN,...,395.0,352.0,1.0,2475.0,10,0.0,0.0,20.0,0.0,1.0
4,2026,1,1,1,4,1/1/2026 12:00:00 AM,AA,19805,AA,N103NN,...,302.0,282.0,1.0,2475.0,10,NaN,NaN,NaN,NaN,NaN


In [4]:
# Remove cancelled/diverted flights (no valid departure delay)
if "CANCELLED" in df.columns:
    df = df[df["CANCELLED"] == 0]
if "DIVERTED" in df.columns:
    df = df[df["DIVERTED"] == 0]


In [5]:
# Targets
df = df.dropna(subset=["DEP_DEL15", "DEP_DELAY"])
df["delay_minutes"] = df["DEP_DELAY"]        # or DEP_DELAY_NEW

In [6]:
# Feature engineering (only info known BEFORE departure)

df["dep_hour"] = df["CRS_DEP_TIME"] // 100
df["arr_hour"] = df["CRS_ARR_TIME"] // 100
df["dep_hour"] = df["dep_hour"].replace(24, 0)
df["arr_hour"] = df["arr_hour"].replace(24, 0)

# Optional extras
df["is_weekend"] = df["DAY_OF_WEEK"].isin([6, 7]).astype(int)
df["route"] = df["ORIGIN"] + "_" + df["DEST"]   # high cardinality; remove if too slow

In [7]:
# Feature Lists

numeric_features = [
    "DAY_OF_MONTH", "dep_hour", "arr_hour",
    "CRS_ELAPSED_TIME", "DISTANCE", "is_weekend",
]
categorical_features = [
    "MONTH_CAT", "DAY_OF_WEEK", "DEP_TIME_BLK",
    "OP_UNIQUE_CARRIER", "ORIGIN", "DEST",
]

# MONTH as category would break with unseen months (train Jan, test Feb),
# so keep MONTH out of the features. It's only used for splitting.
categorical_features.remove("MONTH_CAT")


# weather_numeric = ["temperature", "wind_speed", "precipitation", "visibility"]
# numeric_features += weather_numeric

# Keep only columns that exist
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

In [8]:
# Preprocessing

def make_preprocessor():
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ])

def make_classifier():
    return Pipeline([
        ("prep", make_preprocessor()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])

def make_regressor():
    return Pipeline([
        ("prep", make_preprocessor()),
        ("model", LinearRegression()),
    ])

In [10]:
# Expanding-window splits (train months, validation/test month)

splits = [
    ([1],             2, "validate"),
    ([1, 2],          3, "validate"),
    ([1, 2, 3],       4, "validate"),
    ([1, 2, 3, 4],    5, "validate"),
    ([1, 2, 3, 4, 5], 6, "TEST"),
]

clf_results, reg_results = [], []

for train_months, eval_month, kind in splits:
    train = df[df["MONTH"].isin(train_months)]
    evalset = df[df["MONTH"] == eval_month]

    X_train = train[numeric_features + categorical_features]
    X_eval = evalset[numeric_features + categorical_features]

    label = f"Train {train_months[0]}-{train_months[-1]} -> {kind} {eval_month}"

    # ---------- Classification ----------
    clf = make_classifier()
    clf.fit(X_train, train["DEP_DEL15"])
    pred = clf.predict(X_eval)
    proba = clf.predict_proba(X_eval)[:, 1]
    y = evalset["DEP_DEL15"]
    clf_results.append({
        "split": label,
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred),
        "f1": f1_score(y, pred),
        "roc_auc": roc_auc_score(y, proba),
    })

    # ---------- Regression ----------
    reg = make_regressor()
    reg.fit(X_train, train["delay_minutes"])
    pred_r = reg.predict(X_eval)
    y_r = evalset["delay_minutes"]
    reg_results.append({
        "split": label,
        "MAE": mean_absolute_error(y_r, pred_r),
        "RMSE": np.sqrt(mean_squared_error(y_r, pred_r)),
        "R2": r2_score(y_r, pred_r),
    })

In [11]:
# Results

clf_df = pd.DataFrame(clf_results)

# First 4 rows = validation folds (Feb-May), last row = test (Jun)
cv_df = clf_df.iloc[:4]
test_df = clf_df.iloc[4]

print("\n=== PER-SPLIT CLASSIFICATION RESULTS ===")
print(clf_df.round(4).to_string(index=False))

print("\n=== CROSS-VALIDATION SUMMARY (validation folds: Feb-May) ===")
for m in ["roc_auc", "accuracy", "precision", "recall", "f1"]:
    print(f"CV {m:<10}: {cv_df[m].mean():.4f} (+/- {cv_df[m].std():.4f})")

print("\n=== FINAL TEST RESULTS (June) ===")
for m in ["roc_auc", "accuracy", "precision", "recall", "f1"]:
    print(f"Test {m:<10}: {test_df[m]:.4f}")

print("\n=== REGRESSION RESULTS ===")
print(pd.DataFrame(reg_results).round(4).to_string(index=False))


=== PER-SPLIT CLASSIFICATION RESULTS ===
                  split  accuracy  precision  recall     f1  roc_auc
Train 1-1 -> validate 2    0.5927     0.2759  0.6341 0.3845   0.6515
Train 1-2 -> validate 3    0.5920     0.3377  0.6598 0.4468   0.6550
Train 1-3 -> validate 4    0.5956     0.2853  0.6732 0.4007   0.6704
Train 1-4 -> validate 5    0.6083     0.3203  0.6857 0.4366   0.6838
    Train 1-5 -> TEST 6    0.6418     0.3999  0.6987 0.5087   0.7172

=== CROSS-VALIDATION SUMMARY (validation folds: Feb-May) ===
CV roc_auc   : 0.6652 (+/- 0.0149)
CV accuracy  : 0.5971 (+/- 0.0076)
CV precision : 0.3048 (+/- 0.0291)
CV recall    : 0.6632 (+/- 0.0221)
CV f1        : 0.4172 (+/- 0.0294)

=== FINAL TEST RESULTS (June) ===
Test roc_auc   : 0.7172
Test accuracy  : 0.6418
Test precision : 0.3999
Test recall    : 0.6987
Test f1        : 0.5087

=== REGRESSION RESULTS ===
                  split     MAE    RMSE     R2
Train 1-1 -> validate 2 23.8900 54.5899 0.0101
Train 1-2 -> validate 3 27.289